# 04 — Corriger la normale plutôt que la remplacer

**Constat du notebook 03 :** le boosting a la meilleure MAE, mais il sous-estime durablement le PDQ 21 (biais de +9,4 au test),
alors que la moyenne des 12 dernières semaines (« normale ») suivait déjà son nouveau niveau.

**Question :** un modèle qui part de la normale et n'apprend qu'une **correction** fait-il mieux que la normale,
de façon régulière d'une année à l'autre, sans sous-estimer les PDQ qui changent de volume ?

## Protocole, fixé avant de comparer
- **Mêmes lignes pour tous** : mêmes PDQ, mêmes semaines, même horizon de 4 semaines.
- **Origine glissante** : chaque 1er janvier, chaque candidat est réentraîné sur toutes les semaines déjà consolidées,
  puis prédit l'année suivante. On imite ainsi un service maintenu, pas un modèle figé en 2022.
- **Choix sur 2019–2023** : la meilleure MAE **parmi les candidats dont aucun PDQ n'est biaisé de plus de 2 actes** (`crimes.toml`).
- **2024–2026 a déjà été consulté** (notebooks 02 et 03) : ces années servent au diagnostic, pas au choix.
  Les semaines qui arriveront après la mise en service serviront à confirmer.
- Le rejeu est **rétrospectif** : un seul extrait du fichier ; la consolidation en 14 jours reste une hypothèse.
- La cohorte de 29 PDQ a été choisie **rétrospectivement** (actifs en 2025–2026, notebook 00) ; le PDQ 22 n'y figure pas.

| Candidat | Prévision |
|---|---|
| `moy_12` | moyenne des 12 dernières semaines consolidées (la normale) |
| `mix_4_12` | 50 % moyenne 4 semaines + 50 % moyenne 12 semaines |
| `ridge_residu` | normale + correction apprise par un Ridge (écarts récents à la normale et mois, sans identifiant PDQ) |
| `ridge_hybride` | comme `ridge_residu`, avec en plus l'identifiant PDQ (ajustement par PDQ, tiré vers zéro par la pénalité du Ridge) |
| `ridge_local` | la même recette que `ridge_residu`, entraînée séparément sur chaque PDQ (29 modèles) |
| `boosting_hist` | le modèle du notebook 03, réentraîné de la même façon (référence) |

`ridge_hybride` et `ridge_local` ont été **ajoutés après une revue critique**, une fois les quatre premiers candidats comparés.
La règle de choix n'a pas changé ; mais 2024–2026 avait déjà été consulté avant leur ajout, ce n'est donc pas un test indépendant pour eux.

In [1]:
import json
import hashlib
import tomllib
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
with open(ROOT / "configs/crimes.toml", "rb") as f:
    cfg = tomllib.load(f)

SEED = cfg["experiment"]["seed"]
H = cfg["historique"]
C = cfg["comparaison"]
HORIZON = H["horizon"]
IN = ROOT / "artifacts" / "00_start_here"

panel = pd.read_parquet(IN / "panel.parquet")[["SEMAINE", "PDQ", "y"]]
meta = json.loads((IN / "meta.json").read_text())
meta_03 = json.loads((ROOT / "artifacts" / "03_historique" / "meta.json").read_text())

empreinte = hashlib.sha256((ROOT / cfg["data"]["path"]).read_bytes()).hexdigest()[:16]
assert empreinte == meta["source_sha16"], "Le CSV a changé depuis le notebook 00 : relance le 00 d'abord."
assert meta_03["source_sha16"] == meta["source_sha16"], "Le notebook 03 a été fait sur d'autres données : relance le 03."
assert meta_03["lag_min_semaines"] == HORIZON, "Le 03 n'a pas le même horizon que la config : relance le 03."

print("Panel :", panel.shape, "| horizon :", HORIZON, "semaines")
print("Référence (notebook 03) :", meta_03["modele_retenu"], "| MAE test", meta_03["mae_test"])

Panel : (17893, 3) | horizon : 4 semaines
Référence (notebook 03) : boosting_hist | MAE test 4.686


## Variables
Les mêmes que dans le notebook 03 (`lag_4`, `lag_5`, `lag_52`, `moy_4`, `moy_12`, mois).
Le Ridge sur résidus utilise en plus des **écarts à la normale** : `moy_4 − moy_12` (tendance récente),
`lag_4 − moy_12` (dernière semaine connue) et `lag_52 − moy_12` (même semaine l'an passé).
Sa cible est `y − moy_12` : il apprend **de combien corriger la normale**, pas le niveau du PDQ.

In [2]:
panel = panel.sort_values(["PDQ", "SEMAINE"]).reset_index(drop=True)
par_pdq = panel.groupby("PDQ")["y"]

for k in H["lags"]:
    panel[f"lag_{k}"] = par_pdq.shift(k)
for w in H["fenetres"]:
    panel[f"moy_{w}"] = par_pdq.transform(lambda s: s.shift(HORIZON).rolling(w).mean())

HISTOIRE = [f"lag_{k}" for k in H["lags"]] + [f"moy_{w}" for w in H["fenetres"]]
panel = panel.dropna(subset=HISTOIRE).reset_index(drop=True)

panel["mois"] = panel["SEMAINE"].dt.month
panel["annee"] = panel["SEMAINE"].dt.year
panel["ecart_moy_4"] = panel["moy_4"] - panel["moy_12"]
panel["ecart_lag_4"] = panel[f"lag_{HORIZON}"] - panel["moy_12"]
panel["ecart_lag_52"] = panel["lag_52"] - panel["moy_12"]
ECARTS = ["ecart_moy_4", "ecart_lag_4", "ecart_lag_52"]

assert list(meta_03["colonnes"]) == HISTOIRE + ["PDQ", "mois"], "Les variables ne sont plus celles du notebook 03."
print("Lignes :", len(panel), "|", panel["SEMAINE"].min().date(), "→", panel["SEMAINE"].max().date())
panel.groupby("annee").size()

Lignes : 16385 | 2016-01-04 → 2026-10-26


annee
2016    1508
2017    1508
2018    1537
2019    1508
2020    1508
2021    1508
2022    1508
2023    1508
2024    1537
2025    1508
2026    1247
dtype: int64

## Les candidats
Les deux règles n'apprennent rien. Les deux modèles sont des pipelines scikit-learn ; le Ridge prédit une correction,
qu'on ajoute ensuite à la normale. Les réglages du boosting sont ceux du notebook 03, `alpha=10` pour le Ridge comme au notebook 03 :
rien n'est réglé dans ce notebook.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def pipeline(estimateur, numeriques, categorielles):
    return make_pipeline(ColumnTransformer([
        ("num", StandardScaler(), numeriques),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorielles),
    ]), estimateur)

def ridge_residu():
    return pipeline(Ridge(alpha=10), ECARTS, ["mois"])

def ridge_hybride():
    return pipeline(Ridge(alpha=10), ECARTS, ["mois", "PDQ"])

def boosting_hist():
    return pipeline(HistGradientBoostingRegressor(max_iter=120, max_leaf_nodes=15, learning_rate=0.08,
                                                  min_samples_leaf=20, early_stopping=False, random_state=SEED),
                    HISTOIRE, ["PDQ", "mois"])

def predire(entrainement, a_predire):
    """Les six prévisions pour les lignes à prédire, avec des modèles entraînés sur `entrainement` seulement."""
    prev = pd.DataFrame(index=a_predire.index)
    prev["moy_12"] = a_predire["moy_12"]
    prev["mix_4_12"] = 0.5 * a_predire["moy_4"] + 0.5 * a_predire["moy_12"]

    r = ridge_residu().fit(entrainement, entrainement["y"] - entrainement["moy_12"])
    prev["ridge_residu"] = a_predire["moy_12"] + r.predict(a_predire)

    h = ridge_hybride().fit(entrainement, entrainement["y"] - entrainement["moy_12"])
    prev["ridge_hybride"] = a_predire["moy_12"] + h.predict(a_predire)

    prev["ridge_local"] = np.nan
    for pdq, lignes in a_predire.groupby("PDQ"):
        local = entrainement[entrainement["PDQ"] == pdq]
        m = ridge_residu().fit(local, local["y"] - local["moy_12"])
        prev.loc[lignes.index, "ridge_local"] = lignes["moy_12"] + m.predict(lignes)

    b = boosting_hist().fit(entrainement, entrainement["y"])
    prev["boosting_hist"] = b.predict(a_predire)
    return prev

CANDIDATS = ["moy_12", "mix_4_12", "ridge_residu", "ridge_hybride", "ridge_local", "boosting_hist"]

## Origine glissante
Pour chaque année, on se place le 1er janvier : les dernières cibles consolidées sont celles d'il y a 4 semaines.
On entraîne sur tout ce qui précède, puis on prédit toute l'année, sans réentraîner en cours d'année.

In [4]:
premiere, derniere = C["annees_choix"][0], C["annees_diagnostic"][1]
morceaux = []
for annee in range(premiere, derniere + 1):
    coupure = pd.Timestamp(f"{annee}-01-01") - pd.Timedelta(weeks=HORIZON)
    entrainement = panel[panel["SEMAINE"] < coupure]
    a_predire = panel[panel["annee"] == annee]
    assert entrainement["SEMAINE"].max() < a_predire["SEMAINE"].min() - pd.Timedelta(weeks=HORIZON - 1)
    morceaux.append(predire(entrainement, a_predire))
    print(annee, "| entraînement jusqu'au", entrainement["SEMAINE"].max().date(), "|", len(a_predire), "lignes prédites")

prev = pd.concat(morceaux)
evalue = panel.loc[prev.index, ["SEMAINE", "PDQ", "annee", "y", "moy_12"]]
erreur = prev.rsub(evalue["y"], axis=0)        # réel − prévu : positif = le modèle sous-estime

2019 | entraînement jusqu'au 2018-12-03 | 1508 lignes prédites


2020 | entraînement jusqu'au 2019-12-02 | 1508 lignes prédites


2021 | entraînement jusqu'au 2020-11-30 | 1508 lignes prédites


2022 | entraînement jusqu'au 2021-11-29 | 1508 lignes prédites


2023 | entraînement jusqu'au 2022-11-28 | 1508 lignes prédites


2024 | entraînement jusqu'au 2023-11-27 | 1537 lignes prédites


2025 | entraînement jusqu'au 2024-12-02 | 1508 lignes prédites


2026 | entraînement jusqu'au 2025-12-01 | 1247 lignes prédites


## Choix sur 2019–2023
On calcule sur les années de choix la MAE, le p90 de l'erreur absolue et le pire biais moyen d'un PDQ.
Puis on applique la règle écrite plus haut, **sans regarder 2024–2026**.

In [5]:
choix_ = evalue["annee"].between(*C["annees_choix"])
biais_par_pdq = erreur[choix_].groupby(evalue.loc[choix_, "PDQ"]).mean()

selection = pd.DataFrame({
    "mae": erreur[choix_].abs().mean(),
    "p90_erreur": erreur[choix_].abs().quantile(0.9),
    "pire_biais_pdq": biais_par_pdq.abs().max(),
    "pdq_le_plus_biaise": biais_par_pdq.abs().idxmax(),
    "nb_pdq_biaises": (biais_par_pdq.abs() > C["biais_max_pdq"]).sum(),
})
selection["admissible"] = selection["pire_biais_pdq"] <= C["biais_max_pdq"]
RETENU = selection[selection["admissible"]]["mae"].idxmin()

print("Règle : meilleure MAE parmi les candidats sans PDQ biaisé de plus de", C["biais_max_pdq"], "actes →", RETENU)
selection.round(3)

Règle : meilleure MAE parmi les candidats sans PDQ biaisé de plus de 2.0 actes → ridge_residu


,mae,p90_erreur,pire_biais_pdq,pdq_le_plus_biaise,nb_pdq_biaises,admissible
moy_12,5.039,10.833,0.766,26,0,True
mix_4_12,4.916,10.542,0.571,26,0,True
ridge_residu,4.779,10.247,1.362,26,0,True
ridge_hybride,4.795,10.273,1.294,7,0,True
ridge_local,4.855,10.371,1.187,7,0,True
boosting_hist,4.768,10.265,2.673,26,1,False


In [6]:
mae_par_annee = erreur.abs().groupby(evalue["annee"]).mean().T
mae_par_annee["gain_min_sur_moy_12"] = np.nan
gains = mae_par_annee.loc["moy_12", range(*[C["annees_choix"][0], C["annees_choix"][1] + 1])] - \
        mae_par_annee[range(C["annees_choix"][0], C["annees_choix"][1] + 1)]
mae_par_annee["gain_min_sur_moy_12"] = gains.min(axis=1)
print("MAE par année (actes par PDQ-semaine). Dernière colonne : plus petit gain annuel sur la normale, 2019–2023.")
mae_par_annee.round(2)

MAE par année (actes par PDQ-semaine). Dernière colonne : plus petit gain annuel sur la normale, 2019–2023.


annee,2019,2020,2021,2022,2023,2024,2025,2026,gain_min_sur_moy_12
moy_12,4.57,4.87,5.09,5.63,5.04,4.85,4.99,5.01,0.00
mix_4_12,4.45,4.79,4.87,5.44,5.02,4.83,4.84,4.87,0.02
ridge_residu,4.28,4.69,4.69,5.20,5.03,4.64,4.70,4.74,0.01
ridge_hybride,4.32,4.71,4.70,5.21,5.04,4.64,4.70,4.74,0.00
ridge_local,4.42,4.80,4.90,5.17,4.98,4.61,4.65,4.70,0.06
boosting_hist,4.31,4.76,4.68,5.09,5.00,4.50,4.74,4.65,0.04


## Incertitude du gain
Une MAE plus basse sur un échantillon ne prouve pas que le gain tiendra. Les erreurs d'une même semaine sont corrélées entre PDQ,
et les semaines voisines se ressemblent : on ne peut pas traiter les 1 508 lignes d'une année comme indépendantes.
On calcule donc **un gain par semaine** (moyenne sur les 29 PDQ), puis on rééchantillonne **des blocs de 8 semaines consécutives**
(*bootstrap par blocs*) pour obtenir un intervalle à 95 %.

In [7]:
rng = np.random.default_rng(SEED)
BLOC = 8

def intervalle_gain(modele, reference, masque, tirages=2000):
    par_semaine = (erreur[reference].abs() - erreur[modele].abs())[masque].groupby(evalue.loc[masque, "SEMAINE"]).mean().to_numpy()
    n = len(par_semaine)
    debuts = rng.integers(0, n - BLOC + 1, size=(tirages, int(np.ceil(n / BLOC))))
    moyennes = np.array([np.concatenate([par_semaine[d:d + BLOC] for d in ligne])[:n].mean() for ligne in debuts])
    bas, haut = np.quantile(moyennes, [0.025, 0.975])
    return {"semaines": n, "gain": par_semaine.mean(), "ic95_bas": bas, "ic95_haut": haut,
            "semaines_gagnees": (par_semaine > 0).mean()}

diag_ = evalue["annee"].between(*C["annees_diagnostic"])
lignes = []
for periode, masque in [("choix 2019–2023", choix_), ("diagnostic 2024–2026", diag_)]:
    for modele, reference in [(RETENU, "moy_12"), (RETENU, "boosting_hist"), ("boosting_hist", "moy_12")]:
        if modele != reference:
            lignes.append({"periode": periode, "comparaison": f"{modele} vs {reference}",
                           **intervalle_gain(modele, reference, masque)})
# Ajoutés après une revue critique : les deux candidats par PDQ contre le Ridge global.
# Tirés après les comparaisons d'origine pour ne pas changer leurs intervalles.
for periode, masque in [("choix 2019–2023", choix_), ("diagnostic 2024–2026", diag_)]:
    for modele in ["ridge_hybride", "ridge_local"]:
        lignes.append({"periode": periode, "comparaison": f"{modele} vs {RETENU}",
                       **intervalle_gain(modele, RETENU, masque)})
print("Gain = MAE de la référence − MAE du modèle (positif : le modèle fait mieux).")
pd.DataFrame(lignes).round(3)

Gain = MAE de la référence − MAE du modèle (positif : le modèle fait mieux).


,periode,comparaison,semaines,gain,ic95_bas,ic95_haut,semaines_gagnees
0,choix 2019–2023,ridge_residu vs moy_12,260,0.260,0.122,0.389,0.673
1,choix 2019–2023,ridge_residu vs boosting_hist,260,-0.011,-0.078,0.053,0.469
2,choix 2019–2023,boosting_hist vs moy_12,260,0.271,0.128,0.407,0.650
3,diagnostic 2024–2026,ridge_residu vs moy_12,148,0.260,0.111,0.401,0.642
4,diagnostic 2024–2026,ridge_residu vs boosting_hist,148,-0.056,-0.110,0.020,0.432
5,diagnostic 2024–2026,boosting_hist vs moy_12,148,0.316,0.136,0.468,0.676
6,choix 2019–2023,ridge_hybride vs ridge_residu,260,-0.016,-0.026,-0.006,0.385
7,choix 2019–2023,ridge_local vs ridge_residu,260,-0.076,-0.132,-0.022,0.400
8,diagnostic 2024–2026,ridge_hybride vs ridge_residu,148,-0.001,-0.009,0.006,0.466
9,diagnostic 2024–2026,ridge_local vs ridge_residu,148,0.033,-0.022,0.073,0.554


## Diagnostic sur 2024–2026 (période déjà consultée)
Quatre mesures pour chaque candidat :
- MAE et p90 de l'erreur absolue ;
- biais du PDQ 21 **avant** (2024) et **après** (2025–2026) la fermeture du PDQ 22, qui lui a transféré une partie de son territoire ;
- hausses notables détectées : une hausse est « réel ≥ normale + 5 actes » (seuil fixé dans la config avant la comparaison),
  une alerte est « prévu ≥ normale + 5 ».

In [8]:
d = evalue[diag_]
e = erreur[diag_]
pdq_21 = d["PDQ"] == "21"
hausse = d["y"] - d["moy_12"] >= C["seuil_hausse"]

diagnostic = pd.DataFrame({
    "mae": e.abs().mean(),
    "p90_erreur": e.abs().quantile(0.9),
    "biais_pdq_21_2024": e[pdq_21 & (d["annee"] == 2024)].mean(),
    "biais_pdq_21_2025_26": e[pdq_21 & (d["annee"] >= 2025)].mean(),
    "pire_biais_pdq": e.groupby(d["PDQ"]).mean().abs().max(),
})
for c in CANDIDATS:
    alerte = prev.loc[diag_, c] - d["moy_12"] >= C["seuil_hausse"]
    diagnostic.loc[c, "alertes"] = alerte.sum()
    diagnostic.loc[c, "hausses_detectees"] = (alerte & hausse).sum() / hausse.sum()
    diagnostic.loc[c, "part_fausses_alertes"] = (alerte & ~hausse).sum() / alerte.sum() if alerte.any() else np.nan

print("Hausses notables réelles en 2024–2026 :", int(hausse.sum()), "sur", len(d), "lignes")
diagnostic.round(3)

Hausses notables réelles en 2024–2026 : 777 sur 4292 lignes


,mae,p90_erreur,biais_pdq_21_2024,biais_pdq_21_2025_26,pire_biais_pdq,alertes,hausses_detectees,part_fausses_alertes
moy_12,4.945,10.500,-0.096,1.667,1.035,0.0,0.000,NaN
mix_4_12,4.843,10.375,-0.142,1.344,0.812,27.0,0.023,0.333
ridge_residu,4.686,9.955,0.255,1.361,0.965,48.0,0.041,0.333
ridge_hybride,4.687,9.974,-0.180,0.779,0.917,56.0,0.046,0.357
ridge_local,4.653,10.000,-0.041,0.167,0.948,110.0,0.072,0.491
boosting_hist,4.630,9.878,3.459,7.041,5.758,58.0,0.048,0.362


## Le PDQ 21, année par année
Biais moyen (réel − prévu) : un chiffre positif est une sous-estimation.

In [9]:
pdq21 = evalue["PDQ"] == "21"
tableau_21 = erreur[pdq21].groupby(evalue.loc[pdq21, "annee"]).mean().T
tableau_21.loc["actes_reels_moyens"] = evalue.loc[pdq21].groupby("annee")["y"].mean()
tableau_21.round(1)

annee,2019,2020,2021,2022,2023,2024,2025,2026
moy_12,-0.8,-3.1,2.5,2.8,0.4,-0.1,1.1,2.3
mix_4_12,-0.7,-2.4,1.9,2.0,0.6,-0.1,0.7,2.1
ridge_residu,-0.5,-3.8,2.9,3.4,1.9,0.3,2.4,0.1
ridge_hybride,-1.3,-4.2,3.5,3.5,1.6,-0.2,2.0,-0.7
ridge_local,-1.1,-4.4,3.2,3.2,1.7,-0.0,1.3,-1.1
boosting_hist,-1.7,-6.2,2.4,5.4,5.9,3.5,11.7,1.4
actes_reels_moyens,33.9,22.0,27.9,37.5,48.6,50.2,63.9,55.1


## Pourquoi le boosting reste trop bas en 2025
En 2025, première année après la fermeture du PDQ 22, chaque modèle a été entraîné jusqu'à décembre 2024.
On compare, en moyenne sur les 52 semaines du PDQ 21, la **correction appliquée à la normale** par chaque candidat
à la correction qu'il aurait fallu appliquer (réel − normale).

In [10]:
m21 = (evalue["PDQ"] == "21") & (evalue["annee"] == 2025)
correction = prev[m21].sub(evalue.loc[m21, "moy_12"], axis=0).mean()
pd.DataFrame({"correction_moyenne": correction,
              "correction_necessaire": (evalue.loc[m21, "y"] - evalue.loc[m21, "moy_12"]).mean(),
              "prevision_moyenne": prev[m21].mean(),
              "reel_moyen": evalue.loc[m21, "y"].mean()}).round(1)

,correction_moyenne,correction_necessaire,prevision_moyenne,reel_moyen
moy_12,0.0,1.1,62.8,63.9
mix_4_12,0.4,1.1,63.2,63.9
ridge_residu,-1.3,1.1,61.5,63.9
ridge_hybride,-0.9,1.1,61.9,63.9
ridge_local,-0.1,1.1,62.7,63.9
boosting_hist,-10.6,1.1,52.2,63.9


La normale suivait déjà le nouveau niveau du PDQ 21 ; le boosting l'a corrigée vers le bas. Il connaît l'**identité du PDQ**
et `lag_52` (l'an passé, avant le transfert de territoire) : il ramène le poste vers le niveau qu'il a appris dans le passé.
Le Ridge sur résidus ne connaît ni l'identité ni le niveau du PDQ : il part de la normale et applique une correction
commune à tous les postes. Ses coefficients se lisent directement, en actes de correction par acte d'écart :

In [11]:
coupe = panel["annee"] <= C["annees_choix"][1]
ridge_2023 = ridge_residu().fit(panel[coupe], panel.loc[coupe, "y"] - panel.loc[coupe, "moy_12"])
echelle = ridge_2023[0].named_transformers_["num"].scale_
pd.Series(ridge_2023[-1].coef_[:len(ECARTS)] / echelle, index=ECARTS, name="actes par acte d'écart").round(3)

ecart_moy_4     0.256
ecart_lag_4     0.079
ecart_lag_52    0.130
Name: actes par acte d'écart, dtype: float64

## Modèle final
Le candidat retenu est réentraîné sur **toutes** les semaines consolidées disponibles, comme il le serait en service.
La fiche `meta.json` enregistre aussi la configuration utilisée, pas seulement l'empreinte du CSV.

In [12]:
import joblib

OUT = ROOT / "artifacts" / "04_comparaison"
OUT.mkdir(parents=True, exist_ok=True)

assert RETENU == "ridge_residu", "Le notebook est écrit pour un modèle sur résidus : adapte la sauvegarde si la règle choisit autre chose."
COLONNES = ECARTS + ["mois"]
final = ridge_residu().fit(panel[COLONNES], panel["y"] - panel["moy_12"])
joblib.dump(final, OUT / "modele.joblib")
relu = joblib.load(OUT / "modele.joblib")
np.testing.assert_allclose(relu.predict(panel[COLONNES].tail(10)), final.predict(panel[COLONNES].tail(10)))

selection.round(3).to_csv(OUT / "selection_2019_2023.csv")
mae_par_annee.round(3).to_csv(OUT / "mae_par_annee.csv")
diagnostic.round(3).to_csv(OUT / "diagnostic_2024_2026.csv")
tableau_21.round(2).to_csv(OUT / "pdq_21_par_annee.csv")
pd.DataFrame(lignes).round(3).to_csv(OUT / "incertitude_gain.csv", index=False)

meta_04 = {
    "source_sha16": meta["source_sha16"],
    "configuration": {"historique": H, "comparaison": C, "panel": cfg["panel"]},
    "contrat": "v2 : PDQ x semaine, dimanche soir, semaines passees consolidees (horizon 4)",
    "lag_min_semaines": HORIZON,
    "lags": H["lags"],
    "fenetres": H["fenetres"],
    "colonnes": list(final.feature_names_in_),
    "ecarts": {"ecart_moy_4": "moy_4 - moy_12", "ecart_lag_4": f"lag_{HORIZON} - moy_12", "ecart_lag_52": "lag_52 - moy_12"},
    "marge_erreur": round(float(diagnostic.loc[RETENU, "mae"]), 1),
    "cible": "y - moy_12",
    "prevision": "moy_12 + modele.predict(X)",
    "modele_retenu": RETENU,
    "reference": "moy_12",
    "regle_de_choix": f"meilleure MAE {C['annees_choix']} parmi les candidats sans PDQ biaise de plus de {C['biais_max_pdq']}",
    "evaluation": "origine glissante, reentrainement chaque 1er janvier, rejeu retrospectif",
    "mae_choix": round(float(selection.loc[RETENU, "mae"]), 3),
    "mae_choix_reference": round(float(selection.loc["moy_12", "mae"]), 3),
    "mae_diagnostic": round(float(diagnostic.loc[RETENU, "mae"]), 3),
    "mae_diagnostic_reference": round(float(diagnostic.loc["moy_12", "mae"]), 3),
    "entraine_jusqu_au": panel["SEMAINE"].max().date().isoformat(),
    "periode_diagnostic_deja_consultee": True,
}
(OUT / "meta.json").write_text(json.dumps(meta_04, indent=2, ensure_ascii=False))
sorted(p.name for p in OUT.iterdir())

['diagnostic_2024_2026.csv',
 'incertitude_gain.csv',
 'mae_par_annee.csv',
 'meta.json',
 'modele.joblib',
 'pdq_21_par_annee.csv',
 'selection_2019_2023.csv']

## Résumé

| Candidat | MAE choix 2019–2023 | Pire biais d'un PDQ (2019–2023) | MAE diagnostic 2024–2026 | Biais PDQ 21 (2025–2026) |
|---|---|---|---|---|
| Normale : moyenne 12 semaines | 5,04 | 0,8 | 4,94 | +1,9 |
| Mélange 4/12 semaines | 4,92 | 0,6 | 4,84 | +1,4 |
| **Ridge sur résidus (retenu)** | **4,78** | **1,4** | **4,68** | **+1,5** |
| Ridge hybride (+ identifiant PDQ) | 4,80 | 1,3 | 4,69 | +0,9 |
| Ridge local (29 modèles) | 4,86 | 1,2 | 4,65 | +0,2 |
| Boosting historique (exclu par la règle) | 4,77 | 2,7 | 4,63 | +7,1 |

**Ce qu'on retient.**
- La règle fixée à l'avance retient le **Ridge sur résidus** : le boosting a une MAE à peine plus basse, mais un PDQ biaisé de plus de 2 actes dès 2019–2023.
- Le Ridge sur résidus bat la normale **chaque année de 2019 à 2026** ; le gain est de 0,26 acte par PDQ-semaine, avec un intervalle à 95 % de [0,12 ; 0,39] sur 2019–2023 et de [0,10 ; 0,40] sur 2024–2026 (bootstrap par blocs de 8 semaines).
- L'écart entre le Ridge sur résidus et le boosting n'est **pas distinguable du hasard** : [−0,08 ; +0,05] sur 2019–2023, [−0,11 ; +0,02] sur 2024–2026.
- **Un modèle par PDQ ne fait pas mieux au moment du choix.** Sur 2019–2023, le Ridge local est moins bon que le global de 0,08 acte [IC 95 % : −0,13 ; −0,02], et l'hybride de 0,02 [−0,03 ; −0,01]. Le local perd surtout en 2019–2021, quand chaque PDQ n'a qu'environ 150 semaines d'entraînement, et gagne chaque année depuis 2022.
- Sur 2024–2026, le Ridge local est légèrement meilleur (+0,03 acte) mais l'écart **n'est pas distinguable du hasard** [−0,02 ; +0,07]. Il corrige mieux le PDQ 21 (biais +0,2 au lieu de +1,5), mais émet deux fois plus d'alertes, dont 49 % de fausses (33 % pour le global).
- Le Ridge global reste le modèle servi ; le Ridge local est un **challenger** à confirmer sur les semaines prédites après la mise en service.
- En 2025, le boosting retire en moyenne 10,6 actes à la normale du PDQ 21, alors qu'il fallait en ajouter 1,1 ; le Ridge sur résidus en retire 1,3. Sa prévision s'explique toujours comme « normale + correction ».

**Limites connues.**
- **Rejeu rétrospectif :** un seul extrait du fichier ; la consolidation en 14 jours est une hypothèse, pas une mesure.
- **Cohorte rétrospective :** les 29 PDQ ont été choisis parce qu'ils sont actifs en 2025–2026.
- **2024–2026 déjà consulté :** ces chiffres sont un diagnostic ; la confirmation viendra des semaines prédites après la mise en service.
- **Pas d'outil d'alerte :** sur 770 hausses notables (réel ≥ normale + 5) en 2024–2026, aucun candidat n'en annonce plus de 8 % (Ridge local : 7,3 %). Le modèle prévoit un **niveau** ; il ne détecte pas les pics.
  Avec 20 à 60 actes par semaine, la variation aléatoire d'une semaine (≈ √λ, soit 4 à 8 actes) est du même ordre que le seuil de +5 : la plupart de ces « hausses » sont du bruit qu'aucune prévision à 4 semaines ne peut annoncer.
- **Délai de 4 semaines :** un changement brusque n'entre dans la normale qu'après quatre semaines.